# Live SymFP: custom structures and controlled deformations

Interactive extension of the controlled symmetry-breaking examples in Fig. 3. Custom inputs and deformations are exploratory calculations, not a reproduction of a specific published panel.

Compute parameter-free SymFP directly from coordinates using NumPy and SciPy on the CPU. The implementation follows the PF hybrid construction and spectral scoring; no PyTorch, GPU, checkpoint or training is needed. Use the **SymFP demo** kernel from [the setup instructions](README.md#setup).

Use the bundled example or upload a single-frame XYZ file through Jupyter's file browser and enter its path below. Coordinates are interpreted as a finite, nonperiodic structure (normally in Å); periodic crystals are not handled. The complete graph demo accepts 2–128 atoms. The limit bounds interactive work, not the method's applicability.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "demo" / "data" / "manifest.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the SymFP repository or its demo directory.")
CODE_ROOT = PROJECT_ROOT / "code" if (PROJECT_ROOT / "code" / "utils" / "demo_utils.py").is_file() else PROJECT_ROOT
sys.path.insert(0, str(CODE_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from utils.demo_utils import PALETTE, environment_versions, load_rank_activation
from utils.demo_compute import LiveSymFP, read_xyz, deform_structure

display(environment_versions())
plt.rcParams.update({"font.family": "sans-serif", "font.size": 10, "figure.dpi": 120})


## Choose an input and deformation

Set `SOURCE` to `"example"` or `"xyz"`. Relative XYZ paths are resolved from the repository root, e.g. `demo/my_structure.xyz`. Example choices are `B1_octahedron_tetragonal`, `B2_octahedron_trigonal`, and `B3_prism_axial_chirality`; each starts from its undeformed parent.

`STRETCH` is the fractional z-axis strain; `TWIST` is the rotation in radians at maximum positive z, varying linearly with z; `NOISE` is Cartesian Gaussian standard deviation in coordinate units. These editable deformations are exploratory and need not follow the published trajectories. Set all three to zero to compute only your original geometry. Edit parameters and run all cells again.


In [ ]:
SOURCE = "example"
XYZ_PATH = "demo/my_structure.xyz"
EXAMPLE = "B1_octahedron_tetragonal"
STRETCH = 0.15
TWIST = 0.0
NOISE = 0.0
SEED = 0
MAX_ELL = 6
RADIAL_CHANNELS = 16
RANK_TOL = 1e-4
LS_EPS = 1e-10

if SOURCE == "example":
    trajectory, _, _ = load_rank_activation(EXAMPLE)
    original = trajectory[0].astype(float)
    symbols = ["C"] * len(original)
elif SOURCE == "xyz":
    path = Path(XYZ_PATH).expanduser()
    symbols, original = read_xyz(path if path.is_absolute() else PROJECT_ROOT / path)
else:
    raise ValueError('SOURCE must be "example" or "xyz".')
modified = deform_structure(original, stretch=STRETCH, twist=TWIST, noise=NOISE, seed=SEED)
print(f"Input: {SOURCE}; atoms: {len(symbols)}; elements: {', '.join(sorted(set(symbols)))}")


## Compute live

Atomic numbers enter as scalar node attributes, so element labels are retained. Calculations use float64, natural-parity degrees 0 through `MAX_ELL`, and all directed atom pairs without self edges. No separate odd-scalar diagnostic is added.


In [ ]:
calculator = LiveSymFP(max_ell=MAX_ELL, channels=RADIAL_CHANNELS,
                       rank_tol=RANK_TOL, ls_eps=LS_EPS)
reference = calculator.compute(symbols, original)
changed = calculator.compute(symbols, modified)
display(pd.Series({k: str(v) for k, v in calculator.settings.items()}, name="Calculation settings"))
print(f"Initialization: {calculator.initialization_seconds:.3f} s")
print(f"Original: {reference['seconds']:.3f} s; modified: {changed['seconds']:.3f} s")


## Inspect structures and singular-value spectra

Colours distinguish elements. Coordinates are centered for display. The spectrum shows both geometries; values below 1e-12 are clipped for plotting only. The dashed horizontal line is the absolute numerical-rank tolerance.


In [ ]:
fig = plt.figure(figsize=(10, 4))
elements = sorted(set(symbols))
colours = [plt.get_cmap("tab20")(elements.index(s) % 20) for s in symbols]
centered = [p - p.mean(axis=0) for p in (original, modified)]
radius = max(np.abs(p).max() for p in centered) * 1.15
for i, (p, title) in enumerate(zip(centered, ["Original", "Modified"]), 1):
    ax = fig.add_subplot(1, 2, i, projection="3d")
    ax.scatter(*p.T, c=colours, s=45, edgecolors=PALETTE["ink"], depthshade=False)
    for element in elements:
        ax.scatter([], [], [], color=plt.get_cmap("tab20")(elements.index(element) % 20), label=element)
    ax.set(xlim=(-radius, radius), ylim=(-radius, radius), zlim=(-radius, radius), title=title)
    ax.set_box_aspect((1, 1, 1))
    ax.legend(frameon=False)
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(12, 6), constrained_layout=True)
for k, name in enumerate(reference["channels"]):
    ax = axes.flat[k]
    for result, label, colour, style in [(reference, "Original", PALETTE["teal"], "o-"),
                                          (changed, "Modified", PALETTE["coral"], "s--")]:
        sv = result["spectra"][k]
        ax.plot(np.arange(1, len(sv)+1), np.maximum(sv, 1e-12), style, color=colour, label=label)
    ax.axhline(RANK_TOL, color=PALETTE["grey"], ls=":")
    ax.set(title=name, yscale="log", xlabel="Singular-value index", ylabel="Singular value")
    if k == 0:
        ax.legend(frameon=False)
for ax in list(axes.flat)[len(reference["channels"]):]:
    ax.set_visible(False)
plt.show()


## Numerical ranks and candidate compatibility

LS averages the degree-wise relative tail energy beyond each candidate's fixed-subspace dimension at each degree, including degree 0. `pLS = -log10(LS + LS_EPS)`; larger pLS means greater compatibility, not a calibrated probability.

The table lists the 32 crystallographic candidate groups in a fixed order using Hermann–Mauguin notation (e.g. `m-3m` = Oh, `4/mmm` = D4h, `-6m2` = D3h, `32` = D3). This is not a complete list of all molecular point groups. C1 (`1`) has no rank restriction and therefore zero LS for every input. Low LS is a necessary compatibility check, not proof of exact symmetry or a unique group assignment. Different candidates may share a rank template at the selected degree cutoff. Numerical ranks also depend on tolerance and representation coverage. The live calculation uses float64, so near-zero residuals can differ from the published float32 implementation. Interpret values near the numerical floor alongside the spectra and rank templates.


In [ ]:
display(pd.DataFrame({"channel": reference["channels"],
                      "original rank": reference["ranks"], "modified rank": changed["ranks"]}))
scores = reference["scores"].merge(changed["scores"], on="candidate (Hermann–Mauguin)",
                                   suffixes=(" — original", " — modified"))
with pd.option_context("display.max_rows", 40):
    display(scores.style.format({col: "{:.6g}" for col in scores.columns[1:]}))
print("Observable rank upper bounds (capped by channel capacity):")
with pd.option_context("display.max_rows", 40):
    display(reference["bounds"])
